# Notebook 1: Data Exploration
Explore UCF-Crime dataset statistics and sample frame visualisation.

In [ ]:
import sys; sys.path.insert(0, '..')
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from data.dataset_utils import list_videos, get_video_metadata, UCF_CRIME_CLASSES
from data.video_loader import VideoLoader
from utils.config_loader import get_config

config = get_config('../configs/config.yaml')
print('Config loaded:', config.yolo.model)

In [ ]:
# --- 1. Dataset statistics ---
# Update path to your UCF-Crime location
UCF_ROOT = config.evaluation.ucf_crime_root
videos = list_videos(UCF_ROOT)
print(f'Total videos found: {len(videos)}')

# Class distribution
class_counts = {}
for v in videos:
    cls = os.path.basename(os.path.dirname(v))
    class_counts[cls] = class_counts.get(cls, 0) + 1

df = pd.DataFrame(class_counts.items(), columns=['Class', 'Count']).sort_values('Count', ascending=False)
print(df.to_string(index=False))

In [ ]:
# --- 2. Class distribution bar chart ---
plt.figure(figsize=(14, 5))
plt.bar(df['Class'], df['Count'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('UCF-Crime: Videos per Class')
plt.tight_layout()
plt.savefig('ucf_class_distribution.png', dpi=150)
plt.show()

In [ ]:
# --- 3. Sample video metadata ---
if videos:
    meta = get_video_metadata(videos[0])
    print(f'Sample: {os.path.basename(videos[0])}')
    print(f'  FPS: {meta.fps:.1f}, Duration: {meta.duration_sec:.1f}s, Resolution: {meta.width}x{meta.height}')

In [ ]:
# --- 4. Sample frames from one video ---
if videos:
    loader = VideoLoader(videos[0], config)
    frames = []
    for fi, ts, frame in loader.iter_frames():
        frames.append((fi, ts, frame))
        if len(frames) >= 8:
            break
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, (fi, ts, frame) in zip(axes.flat, frames):
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Frame {fi} | {ts:.1f}s')
        ax.axis('off')
    plt.tight_layout()
    plt.show()